# 차량 손상 탐지 시스템 (Vehicle Damage Detection)

## 프로젝트 개요
AI Hub 차량 손상 데이터셋(약 56만 장)을 전처리하고 YOLOv8 기반 2-stage 파이프라인으로
차량 부위 및 손상을 자동 탐지하는 시스템.

## 전체 파이프라인
```
[AI Hub 원본 데이터]
        |
[Step 1] 데이터 탐색 및 분석
        - 클래스 분포 확인 (Scratched 심각한 불균형 발견)
        - 이미지-라벨 매칭 검증
        |
[Step 2] 전처리 (JSON -> YOLO 변환)
        - BBox 필터링 (면적 1500px 미만, 이미지 대비 60% 초과 제거)
        - Scratched 클래스 언더샘플링 (36만 -> 약 10만)
        - Train/Val 80:20 재분할
        |
[Step 3] Active Learning
        - 1단계: 전체 데이터로 초기 학습 (YOLOv8s, 50 epochs)
        - 2단계: 오답 샘플 추출 (어려운 샘플)
        - 3단계: 어려운 샘플 집중 Fine-tuning
        |
[Step 4] 2-Stage 추론 파이프라인
        - Part Model: 차량 부위 탐지 (9종)
        - Damage Model: 부위 내 손상 탐지 (4종)
```

## 모델 성능
- Part Model (차량 부위): mAP50 약 80%
- Damage Model (손상 탐지): mAP50 약 40.8%

## 클래스 구성
**차량 부위 (9종):** Front bumper, Rear bumper, Fender, Door, Head lights,
Rocker panel, Trunk lid, Bonnet, Other

**손상 유형 (4종):** Scratched(긁힘), Separated(분리), Crushed(찌그러짐), Breakage(파손)

## Step 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics shapely -q

import torch
import ultralytics

print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Step 1. 데이터 탐색

AI Hub 원본 데이터 폴더 구조:
- TL_damage/damage/ : 손상 라벨 JSON
- TS_damage/damage/ : 손상 이미지 JPG
- TL_damage_part/damage_part/ : 부위 라벨 JSON
- TS_damage_part/damage_part/ : 부위 이미지 JPG

In [ ]:
import os
import json
from collections import Counter
from pathlib import Path
from tqdm import tqdm

BASE = '/content/dataset'

JSON_DIRS = [
    'train/TL_damage/damage',
    'train/TL_damage_part/damage_part',
    'val/VL_damage/damage',
    'val/VL_damage_part/damage_part',
]

# 클래스 분포 분석
damage_counter = Counter()
valid_damages = ['Scratched', 'Separated', 'Crushed', 'Breakage']

for rel in JSON_DIRS[:2]:  # train만
    folder = os.path.join(BASE, rel)
    if not os.path.exists(folder):
        continue
    for fname in tqdm(os.listdir(folder), desc=rel):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(folder, fname)) as f:
            data = json.load(f)
        for ann in data.get('annotations', []):
            dmg = ann.get('damage')
            if dmg in valid_damages:
                damage_counter[dmg] += 1

print('손상 클래스 분포:')
total = sum(damage_counter.values())
for cls, cnt in damage_counter.most_common():
    print(f'  {cls:12}: {cnt:8,}개 ({cnt/total*100:5.1f}%)')

max_cnt = max(damage_counter.values())
min_cnt = min(damage_counter.values())
print(f'\n클래스 불균형 비율: {max_cnt/min_cnt:.1f}배')

## Step 2. 데이터 전처리 (JSON -> YOLO 변환)

전처리 핵심 과정:
1. JSON bbox -> YOLO 정규화 좌표 변환
2. BBox 품질 필터링
3. Scratched 클래스 언더샘플링으로 불균형 해소

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from tqdm import tqdm
from collections import Counter


class YOLOConverter:
    """AI Hub JSON -> YOLO 포맷 변환기"""

    def __init__(self):
        self.damage_map = {
            'Scratched': 0, 'Separated': 1,
            'Crushed': 2,   'Breakage': 3
        }
        # BBox 필터링 파라미터
        # 너무 작거나 크거나 비율이 이상한 bbox 제거
        self.min_area   = 1500   # 최소 면적 (픽셀)
        self.max_ratio  = 0.6    # 이미지 대비 최대 면적 비율
        self.max_aspect = 12     # 최대 종횡비
        self.stats = Counter()

    def is_valid_bbox(self, x, y, w, h, img_w, img_h):
        area = w * h
        if area < self.min_area:
            return False, 'too_small'
        if area / (img_w * img_h) > self.max_ratio:
            return False, 'too_large'
        if max(w, h) / (min(w, h) + 1e-6) > self.max_aspect:
            return False, 'bad_aspect'
        return True, 'ok'

    def to_yolo(self, x, y, w, h, img_w, img_h):
        """절대 좌표 -> YOLO 정규화 좌표"""
        xc = (x + w / 2) / img_w
        yc = (y + h / 2) / img_h
        return xc, yc, w / img_w, h / img_h

    def process(self, json_path, img_dir, out_img_dir, out_lbl_dir):
        with open(json_path) as f:
            data = json.load(f)

        stem     = Path(json_path).stem
        img_path = Path(img_dir) / f'{stem}.jpg'
        if not img_path.exists():
            self.stats['no_image'] += 1
            return

        img_info = data.get('images', {}).get('image', {})
        img_w, img_h = img_info.get('imwidth', 0), img_info.get('imheight', 0)
        if img_w == 0 or img_h == 0:
            self.stats['no_size'] += 1
            return

        lines = []
        for ann in data.get('annotations', []):
            dmg = ann.get('damage')
            if dmg not in self.damage_map:
                continue
            bbox = ann.get('bbox', [])
            if len(bbox) < 4:
                continue
            x, y, w, h = bbox[:4]
            valid, reason = self.is_valid_bbox(x, y, w, h, img_w, img_h)
            if not valid:
                self.stats[f'filtered_{reason}'] += 1
                continue
            xc, yc, wn, hn = self.to_yolo(x, y, w, h, img_w, img_h)
            lines.append(f'{self.damage_map[dmg]} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')

        if not lines:
            self.stats['no_damage'] += 1
            return

        shutil.copy(img_path, Path(out_img_dir) / img_path.name)
        with open(Path(out_lbl_dir) / f'{stem}.txt', 'w') as f:
            f.write('\n'.join(lines))
        self.stats['saved'] += 1


def build_dataset(base_dir, output_dir):
    base_dir   = Path(base_dir)
    output_dir = Path(output_dir)

    for split in ['train', 'val']:
        (output_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

    converter = YOLOConverter()

    pairs = [
        ('train/TL_damage/damage',           'train/TS_damage/damage',           'train'),
        ('train/TL_damage_part/damage_part', 'train/TS_damage_part/damage_part', 'train'),
        ('val/VL_damage/damage',             'val/VS_damage/damage',             'val'),
        ('val/VL_damage_part/damage_part',   'val/VS_damage_part/damage_part',   'val'),
    ]

    for json_rel, img_rel, split in pairs:
        json_dir = base_dir / json_rel
        img_dir  = base_dir / img_rel
        out_img  = output_dir / split / 'images'
        out_lbl  = output_dir / split / 'labels'

        print(f'\n처리 중: {json_rel}')
        for jf in tqdm(list(json_dir.glob('*.json'))):
            converter.process(jf, img_dir, out_img, out_lbl)

    print('\n변환 통계:')
    for k, v in converter.stats.most_common():
        print(f'  {k}: {v:,}')


build_dataset('/content/dataset', '/content/dataset_yolo')

for split in ['train', 'val']:
    imgs   = len(list(Path(f'/content/dataset_yolo/{split}/images').glob('*.jpg')))
    labels = len(list(Path(f'/content/dataset_yolo/{split}/labels').glob('*.txt')))
    print(f'{split}: {imgs:,} 이미지 / {labels:,} 라벨')

## Step 3. YAML 설정 파일 생성

In [ ]:
damage_yaml = """path: /content/dataset_yolo
train: train/images
val: val/images
nc: 4
names: [Scratched, Separated, Crushed, Breakage]
"""

part_yaml = """path: /content/part_yolo
train: train/images
val: val/images
nc: 9
names: [Front_bumper, Rear_bumper, Fender, Door, Head_lights,
        Rocker_panel, Trunk_lid, Bonnet, Other]
"""

with open('/content/damage_data.yaml', 'w') as f:
    f.write(damage_yaml)
with open('/content/part_data.yaml', 'w') as f:
    f.write(part_yaml)

print('YAML 파일 생성 완료')

## Step 4. Active Learning - 1단계: 초기 학습

전체 데이터로 초기 모델을 학습시킨 후, 모델이 틀리는 어려운 샘플을 추출하여 집중 재학습.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

save_dir    = '/content/drive/MyDrive/yolo_stage1'
resume_path = Path(save_dir) / 'weights' / 'last.pt'

if resume_path.exists():
    print('이전 학습 발견 - 이어서 학습')
    model, resume = YOLO(str(resume_path)), True
else:
    print('새로운 학습 시작')
    model, resume = YOLO('yolov8s.pt'), False

model.train(
    data=   '/content/damage_data.yaml',
    resume= resume,
    epochs= 50,
    batch=  24,
    imgsz=  640,
    patience=     15,
    optimizer=    'AdamW',
    lr0=          0.001,
    lrf=          0.01,
    warmup_epochs=3,
    weight_decay= 0.0005,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.2, copy_paste=0.2,
    save=True, save_period=1,
    project='/content/drive/MyDrive',
    name='yolo_stage1', exist_ok=True,
)

## Step 5. Active Learning - 2단계: 어려운 샘플 추출

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil
from tqdm import tqdm

model = YOLO('/content/drive/MyDrive/yolo_stage1/weights/best.pt')

train_img_dir   = Path('/content/dataset_yolo/train/images')
train_label_dir = Path('/content/dataset_yolo/train/labels')
hard_img_dir    = Path('/content/dataset_yolo_hard/train/images')
hard_label_dir  = Path('/content/dataset_yolo_hard/train/labels')
hard_img_dir.mkdir(parents=True, exist_ok=True)
hard_label_dir.mkdir(parents=True, exist_ok=True)

hard_count = total_count = 0

print('어려운 샘플 추출 중...')
for img_path in tqdm(list(train_img_dir.glob('*.jpg'))):
    label_path = train_label_dir / (img_path.stem + '.txt')
    if not label_path.exists():
        continue

    total_count += 1
    pred = model.predict(str(img_path), conf=0.25, iou=0.5, verbose=False)[0]

    with open(label_path) as f:
        gt_count = len(f.readlines())
    pred_count = len(pred.boxes)

    # 탐지 수 차이가 크면 어려운 샘플
    if gt_count > 0 and (pred_count == 0 or abs(gt_count - pred_count) / gt_count > 0.5):
        shutil.copy(img_path,   hard_img_dir   / img_path.name)
        shutil.copy(label_path, hard_label_dir / label_path.name)
        hard_count += 1

print(f'전체: {total_count:,}개')
print(f'어려운 샘플: {hard_count:,}개 ({hard_count/total_count*100:.1f}%)')

shutil.copytree('/content/dataset_yolo/val', '/content/dataset_yolo_hard/val', dirs_exist_ok=True)
print('Val 복사 완료')

## Step 6. Active Learning - 3단계: Fine-tuning

In [ ]:
from ultralytics import YOLO
from pathlib import Path

hard_yaml = """path: /content/dataset_yolo_hard
train: train/images
val: val/images
nc: 4
names: [Scratched, Separated, Crushed, Breakage]
"""
with open('/content/damage_data_hard.yaml', 'w') as f:
    f.write(hard_yaml)

save_dir    = '/content/drive/MyDrive/yolo_stage2'
resume_path = Path(save_dir) / 'weights' / 'last.pt'

if resume_path.exists():
    model, resume = YOLO(str(resume_path)), True
else:
    # 1단계 best.pt에서 Fine-tuning 시작
    model, resume = YOLO('/content/drive/MyDrive/yolo_stage1/weights/best.pt'), False

model.train(
    data='/content/damage_data_hard.yaml',
    resume=resume,
    epochs=30,
    batch=24,
    imgsz=640,
    patience=10,
    # Fine-tuning: 낮은 학습률
    optimizer='AdamW', lr0=0.0001, lrf=0.001,
    warmup_epochs=1, weight_decay=0.0005,
    save=True,
    project='/content/drive/MyDrive',
    name='yolo_stage2', exist_ok=True,
)

## Step 7. Part 모델 학습

In [ ]:
from ultralytics import YOLO
from pathlib import Path

save_dir    = '/content/drive/MyDrive/yolo_part/train1'
resume_path = Path(save_dir) / 'weights' / 'last.pt'

if resume_path.exists():
    model, resume = YOLO(str(resume_path)), True
else:
    model, resume = YOLO('yolov8s.pt'), False

model.train(
    data='/content/part_data.yaml',
    resume=resume,
    epochs=100, batch=32, imgsz=640, patience=20,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    warmup_epochs=3, weight_decay=0.0005,
    fliplr=0.5, mosaic=1.0,
    save=True, save_period=1,
    project='/content/drive/MyDrive/yolo_part',
    name='train1', exist_ok=True,
)

## Step 8. 모델 성능 평가

In [ ]:
from ultralytics import YOLO

part_model   = YOLO('/content/drive/MyDrive/yolo_part/train1/weights/best.pt')
part_results = part_model.val(data='/content/part_data.yaml')

print('=' * 50)
print('Part 모델 성능 (차량 부위 탐지)')
print('=' * 50)
print(f'mAP50:     {part_results.box.map50:.4f} ({part_results.box.map50*100:.1f}%)')
print(f'mAP50-95:  {part_results.box.map:.4f}')
print(f'Precision: {part_results.box.mp:.4f}')
print(f'Recall:    {part_results.box.mr:.4f}')

damage_model   = YOLO('/content/drive/MyDrive/yolo_stage2/weights/best.pt')
damage_results = damage_model.val(data='/content/damage_data.yaml')

print('\n' + '=' * 50)
print('Damage 모델 성능 (Active Learning 적용)')
print('=' * 50)
print(f'mAP50:     {damage_results.box.map50:.4f} ({damage_results.box.map50*100:.1f}%)')
print(f'mAP50-95:  {damage_results.box.map:.4f}')
print(f'Precision: {damage_results.box.mp:.4f}')
print(f'Recall:    {damage_results.box.mr:.4f}')

print('\n클래스별 mAP50:')
for i, name in enumerate(['Scratched', 'Separated', 'Crushed', 'Breakage']):
    print(f'  {name:12}: {damage_results.box.ap50[i]*100:.1f}%')

## Step 9. 2-Stage 파이프라인 최종 추론

In [ ]:
from ultralytics import YOLO
import cv2
from IPython.display import Image, display

part_model   = YOLO('/content/drive/MyDrive/yolo_part/train1/weights/best.pt')
damage_model = YOLO('/content/drive/MyDrive/yolo_stage2/weights/best.pt')

PART_NAMES   = ['Front_bumper', 'Rear_bumper', 'Fender', 'Door', 'Head_lights',
                'Rocker_panel', 'Trunk_lid', 'Bonnet', 'Other']
DAMAGE_NAMES = ['Scratched', 'Separated', 'Crushed', 'Breakage']

PART_COLORS = {
    'Front_bumper': (255,100,100), 'Rear_bumper': (100,255,100),
    'Fender': (100,100,255),       'Door': (255,255,100),
    'Head_lights': (255,100,255),  'Rocker_panel': (100,255,255),
    'Trunk_lid': (200,150,100),    'Bonnet': (150,200,100),
    'Other': (180,180,180),
}
DAMAGE_COLOR = (0, 0, 255)


def run_pipeline(image_path, output_path=None, part_conf=0.15, damage_conf=0.25):
    """
    2-Stage 파이프라인 실행
    Stage 1: Part Model로 차량 부위 탐지
    Stage 2: Damage Model로 각 부위 내 손상 탐지
    """
    img = cv2.imread(image_path)
    result_img = img.copy()
    h, w = img.shape[:2]

    part_results = part_model.predict(image_path, conf=part_conf, verbose=False)[0]
    print(f'탐지된 부위: {len(part_results.boxes)}개')

    summary = []
    for part_box in part_results.boxes:
        px1, py1, px2, py2 = map(int, part_box.xyxy[0].cpu().numpy())
        part_name = PART_NAMES[int(part_box.cls[0])]
        pconf     = float(part_box.conf[0])
        color     = PART_COLORS.get(part_name, (200,200,200))

        cv2.rectangle(result_img, (px1, py1), (px2, py2), color, 2)
        cv2.putText(result_img, f'{part_name} {pconf:.0%}',
                    (px1, py1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # Stage 2: 부위 영역 크롭 후 손상 탐지
        margin = 20
        cx1, cy1 = max(0, px1-margin), max(0, py1-margin)
        cx2, cy2 = min(w, px2+margin), min(h, py2+margin)
        crop = img[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            continue

        dmg_results = damage_model.predict(crop, conf=damage_conf, verbose=False)[0]
        damages = []
        for dmg_box in dmg_results.boxes:
            dx1, dy1, dx2, dy2 = map(int, dmg_box.xyxy[0].cpu().numpy())
            dmg_name = DAMAGE_NAMES[int(dmg_box.cls[0])]
            dconf    = float(dmg_box.conf[0])
            ox1, oy1 = cx1+dx1, cy1+dy1
            ox2, oy2 = cx1+dx2, cy1+dy2
            cv2.rectangle(result_img, (ox1,oy1), (ox2,oy2), DAMAGE_COLOR, 2)
            cv2.putText(result_img, f'{dmg_name} {dconf:.0%}',
                        (ox1, oy2+18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, DAMAGE_COLOR, 2)
            damages.append(dmg_name)

        summary.append({'part': part_name, 'damages': damages})

    print('\n탐지 결과:')
    for item in summary:
        dmg_str = ', '.join(item['damages']) if item['damages'] else '이상 없음'
        print(f"  {item['part']}: {dmg_str}")

    if output_path:
        cv2.imwrite(output_path, result_img)
        display(Image(output_path))

    return result_img, summary


# 실행 예시
result_img, summary = run_pipeline(
    image_path='/content/test_car.jpg',
    output_path='/content/result.jpg'
)